# 07 — Model Evaluation

**Goal:** evaluate ranking quality with a per-user temporal holdout. For each eligible user, the latest interaction is held out; earlier interactions form training data. This avoids randomly mixing future interactions into training.

## 1. Imports and data

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

# Keep paths consistent whether the notebook is launched from the project root or 02_Notebooks.
CWD = Path.cwd()
PROJECT_ROOT = CWD.parent if CWD.name == "02_Notebooks" else CWD
DATA_RAW = PROJECT_ROOT / "01_Data" / "Raw"
DATA_PROCESSED = PROJECT_ROOT / "01_Data" / "Processed"
MODELS = PROJECT_ROOT / "03_Models"
VISUALS = PROJECT_ROOT / "05_Visualizations"
for folder in [DATA_RAW, DATA_PROCESSED, MODELS, VISUALS]:
    folder.mkdir(parents=True, exist_ok=True)
print("Project root:", PROJECT_ROOT)

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors
from scipy.sparse import csr_matrix
from sklearn.metrics.pairwise import cosine_similarity
movies = pd.read_pickle(DATA_PROCESSED / "movies_processed.pkl").reset_index(drop=True)
ratings = pd.read_csv(DATA_PROCESSED / "ratings_processed.csv")
ratings["timestamp"] = pd.to_numeric(ratings["timestamp"], errors="coerce")
ratings = ratings.sort_values(["userId","timestamp","movieId"])
print("Ratings available:", len(ratings))

Project root: D:\DataScience\notebooks\CineMatch_Ready_Notebooks
Ratings available: 100836


## 2. Temporal leave-one-out split

Users need at least two ratings. Their latest timestamped rating is held out as the test item. Users with only one interaction cannot contribute to this per-user evaluation.

In [2]:
eligible_counts = ratings.groupby("userId").size()
eligible_users = eligible_counts[eligible_counts >= 2].index
eval_data = ratings[ratings.userId.isin(eligible_users)].copy()
test_idx = eval_data.groupby("userId").tail(1).index
test = eval_data.loc[test_idx].copy()
train = eval_data.drop(index=test_idx).copy()
print("Train interactions:", len(train))
print("Test interactions:", len(test))
print("Evaluated users:", test.userId.nunique())

Train interactions: 100226
Test interactions: 610
Evaluated users: 610


## 3. Build train-only content and collaborative models

To prevent leakage, user histories, rating summaries, and item similarity are built only from the training interactions. Movie metadata is used as content information.

In [3]:
tfidf_eval = TfidfVectorizer(stop_words="english", ngram_range=(1,2), max_features=50000)
X_content = tfidf_eval.fit_transform(movies["content_text"].fillna(""))
nn_eval = NearestNeighbors(metric="cosine", algorithm="brute", n_jobs=-1).fit(X_content)

u_ids = np.sort(train.userId.unique())
m_ids = np.sort(train.movieId.unique())
u_map = {int(x): i for i,x in enumerate(u_ids)}
m_map = {int(x): i for i,x in enumerate(m_ids)}
train_matrix = csr_matrix(
    (train.rating.to_numpy(float),
     (train.userId.map(u_map).to_numpy(), train.movieId.map(m_map).to_numpy())),
    shape=(len(u_ids),len(m_ids))
)
means = np.asarray(train_matrix.sum(axis=1)).ravel()/np.maximum(np.diff(train_matrix.indptr),1)
centered_train = train_matrix.copy().astype(float)
for i in range(centered_train.shape[0]):
    a,b = centered_train.indptr[i], centered_train.indptr[i+1]
    centered_train.data[a:b] -= means[i]
sim_items = cosine_similarity(centered_train.T, dense_output=False).tocsr()
sim_items.setdiag(0); sim_items.eliminate_zeros()
print("Train-only model ready.")

Train-only model ready.


## 4. Define recommendation algorithms for evaluation

The content baseline uses the user's positively rated training movies to find similar metadata. The collaborative baseline uses item-item rating similarity. A popularity baseline is included for context.

In [4]:
movie_row_map = {int(mid): i for i,mid in enumerate(movies.movieId)}
def recommend_content_eval(uid, k=10):
    history = train[(train.userId==uid) & (train.rating>=4.0)].sort_values("rating",ascending=False)
    if history.empty:
        history = train[train.userId==uid]
    scores = {}
    for mid in history.movieId.unique():
        if int(mid) not in movie_row_map: continue
        d, ix = nn_eval.kneighbors(X_content[movie_row_map[int(mid)]],
                                   n_neighbors=min(100,len(movies)))
        for dist, row in zip(d[0],ix[0]):
            cid = int(movies.iloc[int(row)].movieId)
            scores[cid] = max(scores.get(cid,0),float(1-dist))
    seen = set(train.loc[train.userId==uid,"movieId"].astype(int))
    return [mid for mid,_ in sorted(scores.items(),key=lambda x:x[1],reverse=True) if mid not in seen][:k]

def recommend_collab_eval(uid, k=10):
    if uid not in u_map: return []
    ui = u_map[uid]
    a,b = train_matrix.indptr[ui],train_matrix.indptr[ui+1]
    cols, vals = train_matrix.indices[a:b],train_matrix.data[a:b]
    scores = np.zeros(len(m_ids)); weights=np.zeros(len(m_ids))
    for col, rating in zip(cols,vals):
        sims = sim_items[:,col].toarray().ravel()
        scores += sims*rating
        weights += np.abs(sims)
    pred=np.divide(scores,weights,out=np.zeros_like(scores),where=weights>0)
    pred[cols]=-np.inf
    order=np.argsort(pred)[::-1]
    return [int(m_ids[j]) for j in order if np.isfinite(pred[j]) and weights[j]>0][:k]

popular_ids = (train.groupby("movieId").rating.count().sort_values(ascending=False).index.astype(int).tolist())
def recommend_popular_eval(uid, k=10):
    seen=set(train.loc[train.userId==uid,"movieId"].astype(int))
    return [m for m in popular_ids if m not in seen][:k]

## 5. Ranking metrics

Precision@K is the fraction of recommended items that are relevant. Recall@K is the fraction of held-out relevant items retrieved. NDCG@K rewards relevant items ranked higher. Hit Rate@K records whether at least one relevant item appears.

In [5]:
def ranking_metrics(recommendations, relevant_items, k=10):
    recs = recommendations[:k]
    relevant = set(relevant_items)
    hits = [1 if item in relevant else 0 for item in recs]
    precision = sum(hits)/k
    recall = sum(hits)/len(relevant) if relevant else 0.0
    f1 = 2*precision*recall/(precision+recall) if precision+recall else 0.0
    dcg = sum(hit/np.log2(rank+2) for rank,hit in enumerate(hits))
    ideal_hits = min(len(relevant),k)
    idcg = sum(1/np.log2(rank+2) for rank in range(ideal_hits))
    ndcg = dcg/idcg if idcg else 0.0
    hit_rate = float(any(hits))
    return {"precision@k":precision,"recall@k":recall,"f1@k":f1,
            "ndcg@k":ndcg,"hit_rate@k":hit_rate}

## 6. Evaluate across held-out users

A single held-out movie per user makes the per-user recall either zero or one. Results depend on the chosen relevance definition; here the held-out rating must be at least 4.0 to count as relevant.

In [ ]:
K=10
methods={"Content-Based":recommend_content_eval,
         "Collaborative":recommend_collab_eval,
         "Popularity":recommend_popular_eval}
rows=[]
for row in test.itertuples():
    uid=int(row.userId)
    relevant={int(row.movieId)} if float(row.rating)>=4.0 else set()
    if not relevant:
        continue
    for method,fn in methods.items():
        recs=fn(uid,K)
        metrics=ranking_metrics(recs,relevant,K)
        rows.append({"method":method,"userId":uid,**metrics,
                     "recommended_count":len(recs)})
results=pd.DataFrame(rows)
if results.empty:
    print("No held-out ratings met the relevance threshold. Review the split or relevance definition.")
    summary=pd.DataFrame()
else:
    summary=results.groupby("method")[["precision@k","recall@k","f1@k","ndcg@k","hit_rate@k"]].mean()
    display(summary)

## 7. Coverage and diversity

Coverage measures the share of catalogue movies recommended at least once. Diversity here is the average pairwise genre Jaccard distance within each recommendation list; higher values indicate less genre overlap.

In [ ]:
recommendation_lists={}
for uid in test.userId.unique():
    recommendation_lists[int(uid)] = recommend_collab_eval(int(uid),K)

recommended_union=set(m for lst in recommendation_lists.values() for m in lst)
catalogue_ids=set(movies.movieId.astype(int))
coverage=len(recommended_union)/len(catalogue_ids) if catalogue_ids else 0
genre_map={int(r.movieId):set(r.genre_list) for r in movies.itertuples()}
def list_diversity(movie_ids):
    distances=[]
    for i in range(len(movie_ids)):
        for j in range(i+1,len(movie_ids)):
            a,b=genre_map.get(movie_ids[i],set()),genre_map.get(movie_ids[j],set())
            union=a|b
            distances.append(1-len(a&b)/len(union) if union else 0)
    return float(np.mean(distances)) if distances else 0.0
diversity=np.mean([list_diversity(x) for x in recommendation_lists.values() if len(x)>1]) if recommendation_lists else 0.0
print(f"Collaborative catalogue coverage: {coverage:.4f}")
print(f"Average intra-list genre diversity: {diversity:.4f}")

## 8. Visualize comparison and save evaluation outputs

In [ ]:
if not summary.empty:
    summary.plot(kind="bar",figsize=(11,5),title="Recommendation Model Comparison")
    plt.ylabel("Metric value"); plt.xticks(rotation=0); plt.tight_layout(); plt.show()
    summary.to_csv(DATA_PROCESSED/"evaluation_summary.csv")
    results.to_csv(DATA_PROCESSED/"evaluation_per_user.csv",index=False)
    print("Saved evaluation_summary.csv and evaluation_per_user.csv")
else:
    print("No evaluation files saved because no relevant test rows were available.")

## Evaluation limitations

Offline metrics are proxies for real user satisfaction. The split and relevance threshold influence the results; a production evaluation would also consider novelty, business constraints, user feedback, and online experiments. Do not claim that one model is best unless the measured results support it.

Next: **08_Movie_Recommendation_Testing.ipynb**.